In [1]:
from __future__ import annotations

import operator
from typing import Annotated, List, TypedDict

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
class Task(BaseModel):
    id: int

    title: str

    goal: str = Field(
        ...,
        description=(
            "One sentence describing what the reader should be able "
            "to do or understand after this section."
        ),
    )

    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description=(
            "3-5 concise, non-overlapping subpoints to cover in this section."
        ),
    )

    target_words: int = Field(
        ...,
        ge=120,
        le=450,
        description="Target word count for this section (120-450).",
    )

    section_type: Literal[
        "intro",
        "core",
        "examples",
        "checklist",
        "common_mistakes",
        "conclusion",
    ] = Field(
        ...,
        description="Use 'common_mistakes' exactly once in the plan.",
    )

In [3]:
class Plan(BaseModel):
    blog_title:str
    audience:str=Field(...,description="Who this blog is for.")
    tone:str=Field(...,description="Writing tone, e.g. practical, crisp, beginner-friendly.")
    tasks:List[Task]

In [4]:
class State(TypedDict):
    topic:str
    plan:Plan
    sections:Annotated[List[str],operator.add]
    final:str

In [11]:
from dotenv import load_dotenv
load_dotenv()
llm=ChatOpenAI(model="gpt-4.1-mini",temperature=0)

In [ ]:
def orchestrator(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    plan = planner.invoke([
        SystemMessage(
            content="""
            You are an expert content strategist and technical research planner.

            Create a practical, structured plan for a high-quality blog based on
            the user's topic.

            Consider:
            - Target audience and their knowledge level
            - The reader's desired outcome
            - Concepts that must be explained
            - Logical progression from fundamentals to advanced ideas
            - Practical examples and real-world use cases
            - Common mistakes and limitations

            The plan must be:
            - Practical and specific
            - Logically ordered
            - Non-repetitive
            - Detailed enough for another agent to write the complete article
            - Focused on providing real value to the reader

            For every task, define:
            - What the section should accomplish
            - What the reader should learn
            - The key points that must be covered

            Return ONLY the structured output matching the Plan schema.
            """
        ),
        HumanMessage(
            content=f"Topic: {state['topic']}"
        )
    ])

    return {"plan": plan}

In [ ]:
def fanout(state:State):
    return [
        Send(
            "worker",
            {"task":task,"topic":state["topic"],state["plan"]}
        )
        for task in state["plan"].tasks
    ]

In [2]:
def worker(payload: dict) -> dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    section_md = llm.invoke(
        [
            SystemMessage(
                content="""
You are a professional technical blog writer working as part of a
multi-agent writing system.

Your job is to write ONLY the assigned section of the blog.

Follow the provided task goal and bullet points closely. Do not write
other sections of the blog and do not repeat information that belongs
to other sections.

Writing requirements:
- Write clear, accurate, and engaging Markdown.
- Use the task title as the section heading.
- Explain concepts progressively and assume the reader has the
  knowledge level specified by the overall plan.
- Expand the provided bullet points with useful explanations rather
  than simply repeating them.
- Include practical examples when they improve understanding.
- Use code examples when the topic genuinely requires them.
- Avoid unnecessary filler, repetition, and overly generic statements.
- Keep the section focused on its specific goal.
- Make the content useful enough to be published with minimal editing.

Return ONLY the Markdown content for this section.
"""
            ),
            HumanMessage(
                content=f"""
Blog topic:
{topic}

Overall blog plan:
{plan}

Current section:
{task.title}

Section goal:
{task.goal}

Key points to cover:
{bullets_text}
"""
            )
        ]
    )

    return {"sections": [section_md.content]}

In [ ]:
def reducer(state: State) -> dict:
    plan = state["plan"]
    sections = state.get("sections", [])

    title = plan.blog_title.strip()

    # Remove empty sections and normalize whitespace
    cleaned_sections = [
        section.strip()
        for section in sections
        if section and section.strip()
    ]

    body = "\n\n".join(cleaned_sections)

    final_md = f"# {title}\n\n{body}"

    return {"final": final_md}

In [ ]:
g = StateGraph(State)

g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")

g.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"]
)

g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

app

In [ ]:
out=app.invoke({"topic":"Write a blog on self attention","sections":[]})